# Training — YOLOv8s (laptop-safe)

Fine-tunes `yolov8s.pt` on the assembled dataset.

## Laptop-friendly defaults (GTX 1060 6GB)

| param | value | why |
|---|---|---|
| base model | `yolov8s.pt` | yolov8m needs ~10 GB VRAM @ batch=16 — won't fit on 6 GB |
| batch | 8 | leaves ~2 GB free; falls back to 4 then 2 on OOM |
| imgsz | 640 | matches the letterboxed frames |
| workers | 2 | fewer CPU data-loaders → laptop stays responsive |
| cache | False | streaming from disk — no RAM thrash |
| amp | True | fp16 mixed precision → ~halves VRAM use |
| save_period | 10 | checkpoint `epoch{N}.pt` every 10 epochs |
| patience | 20 | early-stop after 20 epochs of no val improvement |

If your card has ≥10 GB VRAM, bump `base_model='yolov8m.pt'` and `batch=16` in
the launch cell below.

## 1 · Environment check

In [ ]:
import sys, platform
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch, ultralytics, cv2, numpy as np
print(f'Python              : {platform.python_version()}')
print(f'OS                  : {platform.platform()}')
print(f'torch               : {torch.__version__}')
print(f'ultralytics         : {ultralytics.__version__}')
print(f'opencv              : {cv2.__version__}')
print(f'CUDA available      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version        : {torch.version.cuda}')
    print(f'GPU                 : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory (GB)     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}')


## 2 · Load dataset config & class distribution

In [ ]:
import yaml
from collections import Counter

CONFIG_PATH = ROOT / 'configs' / 'dataset.yaml'
DATASET_DIR = ROOT / 'data' / 'dataset'
cfg = yaml.safe_load(CONFIG_PATH.read_text())
print('dataset.yaml:')
for k, v in cfg.items(): print(f'  {k}: {v}')

names = cfg['names']
print('\nbox counts per split:')
for split in ('train', 'val', 'test'):
    lbl_dir = DATASET_DIR / 'labels' / split
    c: Counter = Counter()
    for f in lbl_dir.rglob('*.txt'):
        for line in f.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                c[int(parts[0])] += 1
    pretty = {names[k]: v for k, v in c.items()}
    n_imgs = sum(1 for _ in (DATASET_DIR / 'images' / split).rglob('*.jpg'))
    print(f'  {split:6s}: {n_imgs:5d} images   boxes={pretty}')


## 3 · Launch training

In [ ]:
# Calls src/train.py's train() with laptop-safe overrides.
# Edit any param here without touching src/train.py.
from src.train import train

results = train(
    data=CONFIG_PATH,
    base_model='yolov8s.pt',      # 6GB-friendly  (use 'yolov8m.pt' on 10GB+ cards)
    epochs=100,
    patience=20,                  # stop early if val mAP stalls for 20 epochs
    imgsz=640,
    batch=8,                      # halves automatically on CUDA OOM
    workers=2,                    # low → keeps the laptop responsive
    save_period=10,               # checkpoint every 10 epochs
    cache=False,                  # do NOT cache to RAM on a laptop
    amp=True,                     # mixed precision = less VRAM + faster
    device='cuda:0',
)


## 4 · Plot training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

RUN_DIR = ROOT / 'runs' / 'fire_smoke_person'
csv = RUN_DIR / 'results.csv'
if not csv.exists():
    candidates = sorted((ROOT / 'runs').rglob('results.csv'))
    csv = candidates[-1] if candidates else csv

df = pd.read_csv(csv)
df.columns = [c.strip() for c in df.columns]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col in [c for c in df.columns if 'loss' in c.lower()]:
    axes[0].plot(df['epoch'], df[col], label=col)
axes[0].set_title('losses'); axes[0].legend(fontsize=8)

for col in [c for c in df.columns if 'mAP50' in c and 'mAP50-95' not in c]:
    axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('mAP@0.5'); axes[1].legend(fontsize=8)

for col in [c for c in df.columns if 'mAP50-95' in c]:
    axes[2].plot(df['epoch'], df[col], label=col)
axes[2].set_title('mAP@0.5:0.95'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()


## 5 · Evaluate on the test split

In [ ]:
from ultralytics import YOLO
import shutil

BEST = ROOT / 'models' / 'best.pt'
if not BEST.exists():
    cand = sorted((ROOT / 'runs').rglob('best.pt'))
    if cand:
        BEST.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cand[-1], BEST)
        print('Copied best.pt from', cand[-1])

model = YOLO(str(BEST))
metrics = model.val(data=str(CONFIG_PATH), split='test', imgsz=640, device='cuda:0')
print(metrics)


## 6 · 20-image prediction grid

In [ ]:
import random, cv2

test_imgs = sorted((DATASET_DIR / 'images' / 'test').rglob('*.jpg'))
sample = random.sample(test_imgs, min(20, len(test_imgs)))

rows = 5; cols = 4
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
for ax, img_path in zip(axes.ravel(), sample):
    res = model.predict(str(img_path), conf=0.3, iou=0.5, verbose=False)[0]
    annotated = res.plot()  # BGR
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=7); ax.axis('off')
for ax in axes.ravel()[len(sample):]: ax.axis('off')
plt.tight_layout(); plt.show()


---

**Next step:** `python src/inference.py --source 0` for live webcam, or `--source path/to/video.mp4 --save` to annotate a file.